# Facet: run the image-to-3D backend on a free GPU

This notebook runs the [Image-to-3D-mesh](https://github.com/Kareemwafaa/Image-to-3D-mesh) backend
(U²-Net segmentation → Zero123++ multi-view diffusion → InstantMesh reconstruction → cleanup → texture baking;
the classic NeuS reconstruction is still available)
on the free GPU that **Google Colab** or **Kaggle** gives you, and makes it reachable from the
internet through a free Cloudflare tunnel. Then you use it from the web app.

**Before you start, turn on the GPU:**

| | Colab | Kaggle |
| --- | --- | --- |
| GPU | *Runtime → Change runtime type → T4 GPU* | Right sidebar → *Session options → Accelerator → GPU T4 x2* |
| Internet | on by default | Right sidebar → *Internet → On* (needs a phone-verified account) |

Then run the cells from top to bottom (*Run all* works). The first run takes about 10 to 15 minutes,
mostly downloading the models (about 10 GB). The last cell prints a link that opens the app already connected to
your GPU. It keeps running until you stop it, and the link stops working when you do.

In [ ]:
# Settings. The defaults are fine; change them only if you use a fork.
REPO_URL = "https://github.com/Kareemwafaa/Image-to-3D-mesh"
BRANCH = "main"
FRONTEND_URL = "https://kareemwafaa.github.io/Image-to-3D-mesh/"

## 1. Check the GPU

In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU found. Colab: Runtime > Change runtime type > T4 GPU. "
        "Kaggle: Session options > Accelerator > GPU T4 x2. Then run this cell again."
    )
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout)

## 2. Get the code

In [ ]:
import os

REPO_DIR = "/content/facet" if os.path.isdir("/content") else "/kaggle/working/facet"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

## 3. Install the backend

Installs the pinned Python packages into their own environment (`/tmp/facet-env`), so the
notebook's preinstalled packages can't clash with them. Takes 1 to 3 minutes.

At the end it checks that every package each stage needs can be imported, that the GPU is
visible, and downloads InstantMesh's model code. If anything is missing it stops here with a
`FAIL` line naming the package, instead of failing later after the big model download.

In [ ]:
!bash {REPO_DIR}/scripts/setup_gpu_env.sh

## 4. Start the server

Starts the API, loads the models (the first time, it downloads about 10 GB of weights),
opens the tunnel and prints:

* **Public API URL**: paste it into the app's *Server* dialog, or
* **Open the app, already connected**: click it.

Keep this cell running while you use the app. Stop it with the ■ button to free the GPU.

In [ ]:
!/tmp/facet-env/bin/python {REPO_DIR}/scripts/serve_public.py --frontend {FRONTEND_URL}

### Troubleshooting

* **"No GPU found"**: turn on the GPU (see the top), then *Run all* again.
* **"Could not open a Cloudflare tunnel"**: on Kaggle, switch *Internet* on. Then run the last cell again.
* **The app says "Server offline"**: the cell must still be running. The URL changes every time
  the cell restarts, so use the newest one.
* **A `FAIL` line in step 3**: the line names the missing package. Run step 3 again; if it
  still fails, open an issue with that output.
* **"CUDA out of memory"**: add `%env INSTANTMESH_VARIANT=base` (smaller model) in a cell before
  step 4, or `%env RECONSTRUCTOR=neus` and `%env PRELOAD_INSTANTMESH=false` for the classic mode.
* **Out of GPU time**: Kaggle gives about 30 GPU hours a week, Colab a varying daily amount.
  Stop the cell when you're done.